In [2]:
import tensorflow as tf
import os
import torch.nn as nn
import numpy as np
import torch

2026-03-09 15:15:27.600996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773069327.786518      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773069327.843358      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773069328.275122      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773069328.275159      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773069328.275162      55 computation_placer.cc:177] computation placer alr

Implement (using existing code) an LSTM model
Train it (use holdout val) over PTB data set
Vary num units and num of stacked LSTM layers
Obtain a test perplexity of 80 or below (lower = better)

In [3]:
url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/"
files = ["ptb.train.txt", "ptb.valid.txt", "ptb.test.txt"]

for f in files: 
    tf.keras.utils.get_file(f, url + f)

def load_data(filename):
    path = os.path.join(os.path.expanduser('~'), '.keras/datasets', filename)
    with open(path, 'r', encoding = 'utf-8') as f:
        return f.read().replace('\n', '<eos>')

train_text = load_data('ptb.train.txt')
valid_text = load_data('ptb.valid.txt')
test_text = load_data('ptb.test.txt')


5101618/5101618 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
399782/399782 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
449945/449945 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
import torch

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [6]:
#More LSTM can be seen in lecture 3
#Use AWD-LSTMSs
#Get unique tokens
tokens = train_text.split()
vocab = sorted(list(set(tokens)))

valid_tok = valid_text.split()
valid_vocab = sorted(list(set(valid_tok)))

#Create mappings
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

vocab_size = 10000
seq_length = 38
step = 1

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(0, len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(tokens, seq_length)
X_valid, y_valid = create_sequences(valid_tok, seq_length)

In [7]:
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

In [8]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size) #converts word IDs to vectors. vocab_size = num of distinct tokens. Embed_size = how big each word vector is 
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
    def forward(self, x, h): #x:(batch_size, seq_length), h: (h0, c0), h0: zeros in shape (num_layers, batch_size, hidden_size), c0: same
        x = self.embed(x)
        out, (h,c) = self.lstm(x, h)
        out = out.reshape(out.size(0)*out.size(1), out.size(2))
        out = self.linear(out)
        return out, (h, c)

embed_size = 128
hidden_size = 5
num_layers = 2
learning_rate = 0.001


model = LSTMModel(vocab_size, embed_size, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [ ]:
print(X_train[0])
print(y_train[0])
X_train_tensor = torch.tensor([[word2idx[w] for w in sentence] for sentence in X_train]).to(device)
y_train_tensor = torch.tensor([word2idx[w] for w in y_train]).to(device)

X_valid_tensor = torch.tensor([[word2idx[w] for w in sentence] for sentence in X_valid]).to(device)
y_valid_tensor = torch.tensor([word2idx[w] for w in y_valid]).to(device)

#Training
num_epochs = 10

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    num_bath = 0
    for batch_idx in range(0, len(X_train_tensor), seq_length):
        X_batch = X_train_tensor[batch_idx:batch_idx+seq_length]
        y_batch = y_train_tensor[batch_idx:batch_idx+seq_length]

        h0 = torch.zeros(num_layers, X_batch.size(0), hidden_size, device=device)
        c0 = torch.zeros(num_layers, y_batch.size(0), hidden_size, device=device)

        pred, _ = model(X_batch, (h0,c0))
        pred_reshaped = pred.reshape(X_batch.size(0), X_batch.size(1), -1)
        pred_last = pred_reshaped[:, -1, :]
        y_flattened = y_batch.view(-1)
        loss = criterion(pred_last, y_flattened)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_bath += 1
    
    model.eval()
    val_total_loss = 0
    val_num_batchs = 0
    with torch.no_grad():
        for val_batch_idx in range(0, len(X_valid_tensor), seq_length):
            x_val_batch = X_valid_tensor[val_batch_idx:val_batch_idx+seq_length]
            y_val_batch = y_valid_tensor[val_batch_idx: val_batch_idx+ seq_length]
        
            h0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)
            c0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)

            predicted_val, _ = model(x_val_batch, (h0_val, c0_val))

            predicted_val_reshaped = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
#y_valid_flat = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
            pred_last_val = predicted_val_reshaped[:, -1, :]

            y_valid_flat = y_val_batch.view(-1)
            
        
            #_, pred_val_val = torch.max(predicted_val, 1)
            val_loss = criterion(pred_last_val, y_valid_flat)
            val_total_loss += val_loss.item()
            val_num_batchs += 1
    avg_train_loss = total_loss / num_bath
    avg_val_loss = val_total_loss / val_num_bath
    val_perplexity = torch.exp(torch.tensor(avg_val_loss)).item()
    print(f"Epoch {epoch}, loss = {avg_train_loss:.4f}, val_perplexity = {val_perpexity:.2f}")

Implement a smoothed trigram model
P_{tri} (w_i | w_{i-2}, w_{i-1}) = c_1 P(w_i | w_{i-2}, w_{i-1}) + c2 P (w_i | w_{i-1}) + (1- c_1 -c_2)P(w_i)
Then optimize the two smoothing parameters by trial and error in order to get the lowest validation perplexity (next measure the test perplexity)

In [ ]:

def predict_next(model, word, sentence, word2idx):
    model.eval()
    h0_predict = torch.zeros(num_layers, 1, hidden_size, device=device)
    c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
    indices = [int(word2idx[w]) for w in sentence]
    tensor_indx_x = torch.tensor(indices, dtype=torch.long, device=device).unsqueeze(0)
    val_per_time_test, _ = model (tensor_indx_x, (h0_predict, c0_predict))
    print(val_per_time_test[-1])
    probs_next_word = torch.softmax(val_per_time_test[-1], dim=-1)
    correct_next_word = word
    correct_next_word_id = word2idx[correct_next_word]
    print(val_per_time_test)
    prob_correct = probs_next_word[correct_next_word_id]
    #predicted_next_word_id = softmax(probs_next_word)
    #predicted_next_word = idx2word[predicted_next_word_id]
    #print(f"Correct word is {word}, predicted one is {predicted_next_word}")
    return float(prob_correct.item())
print(predict_next(model, y_train[0], X_train[0], word2idx))
def calculate_perplexity(word, sentence):
    # PP = (P (tn | t1,..., t_n-1) ... P(t_3|t_1, t_2)P(t_2|t_1)P(t_1))^{-1/n}
    n = len(words)
    p_inside = 1
    for i in range(1, len(words)-1):
        p_inside *= predict_next(model, words[i], words[:i],word2idx)
    PP = (p_inside)**(-1/n)
print(f"x_train[0] is ", X_train[0], X_train[1], X_train[2])
print(f"y_train[0] is {y_train[0]} and {y_train[1]}, {y_train[2]}")

